# Store Sales - Time Series Forecasting baseline

This notebook mirrors the local baseline: lag/rolling sales features, store metadata, oil, holiday indicators, recursive holdout scoring, then final submission generation.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from competitions.store_sales_time_series_forecasting.models.baseline import (
    build_dataset,
    discover_competition_files,
    fit_and_score_holdout,
    fit_final_model,
    generate_submission,
)


In [ ]:
raw_dir = PROJECT_ROOT / 'competitions' / 'store_sales_time_series_forecasting' / 'data' / 'raw'
files = discover_competition_files(raw_dir)
dataset = build_dataset(files)

selection, holdout_metrics, holdout_predictions = fit_and_score_holdout(
    dataset=dataset,
    holdout_days=16,
    recent_train_start='2016-01-01',
    max_train_rows=1_200_000,
    candidate_strategies=('seasonal_naive', 'lightgbm', 'xgboost'),
    seed=42,
)
selection, holdout_metrics


In [ ]:
holdout_predictions.head()

In [ ]:
final_model = fit_final_model(
    dataset=dataset,
    selection=selection,
    recent_train_start='2016-01-01',
    max_train_rows=1_200_000,
    seed=42,
)
submission = generate_submission(final_model, dataset)
submission.head()

In [ ]:
output_path = PROJECT_ROOT / 'competitions' / 'store_sales_time_series_forecasting' / 'submissions' / 'submission.csv'
output_path.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(output_path, index=False)
print(output_path)
